In [1]:
import os
import pandas as pd
import torch
import torch.nn as nn
import timm
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))

torch.backends.cudnn.benchmark = True

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [3]:
class AIGenDetDataset(Dataset):

    def __init__(self, shard_path, transform=None):

        self.shard_path = shard_path
        self.transform = transform

        self.labels = pd.read_csv(
            os.path.join(shard_path, "labels.csv")
        )

        print(f"Loaded {len(self.labels)} images")


    def __len__(self):
        return len(self.labels)


    def __getitem__(self, idx):

        row = self.labels.iloc[idx]

        img_path = os.path.join(
            self.shard_path,
            "images",
            row["image_name"]
        )

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(row["label"]).long()

        return image, label

In [4]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(380),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),
    transforms.ColorJitter(0.1,0.1,0.1,0.05),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize(400),
    transforms.CenterCrop(380),
    transforms.ToTensor(),
])

In [5]:
# DATASET_PATH = "/mnt/c/Development/ai-gen-detection/shard_0/shard_0"
DATASET_PATH = "/home/daniel/datasets/ai-gen/mnt/c/Development/ai-gen-detection/shard_0/shard_0"

dataset = AIGenDetDataset(
    DATASET_PATH
)

Loaded 50000 images


In [6]:
img, label = dataset[0]

print("Image shape:", img.size)
print("Label:", label)

Image shape: (3456, 2736)
Label: tensor(0)


In [12]:
import time

start = time.time()

for i in range(2000):
    _ = dataset[i]

print("Time:", time.time() - start)

Time: 136.3762423992157


In [7]:
# Create custom dataset classes for train and val with proper transforms
class TransformedDataset(torch.utils.data.Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
    
    def __getitem__(self, idx):
        image, label = self.subset[idx]
        if self.transform:
            image = self.transform(image)
        return image, label
    
    def __len__(self):
        return len(self.subset)

indices = list(range(len(dataset)))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.1,
    random_state=42,
    stratify=dataset.labels["label"]
)

# Create subsets and wrap with proper transforms
train_subset = torch.utils.data.Subset(dataset, train_idx)
val_subset = torch.utils.data.Subset(dataset, val_idx)

train_dataset = TransformedDataset(train_subset, train_transform)
val_dataset = TransformedDataset(val_subset, val_transform)

In [8]:
BATCH_SIZE = 16  # Optimized for RTX 4060 laptop (8GB VRAM)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,      # Set to 0 to avoid WSL2 shared memory issues
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE * 2,  # Can use larger batch for validation (no gradients)
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [ ]:
%pip uninstall -y numpy
%pip install numpy==1.26.4

Found existing installation: numpy 2.2.6
Uninstalling numpy-2.2.6:
  Successfully uninstalled numpy-2.2.6
  Obtaining dependency information for numpy==1.26.4 from https://files.pythonhosted.org/packages/4b/d7/ecf66c1cd12dc28b4040b15ab4d17b773b87fa9d29ca16125de01adb36cd/numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 8.4 MB/s eta 0:00:0000:01m00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


In [9]:
model = timm.create_model(
    "efficientnet_b4",
    pretrained=True,
    num_classes=2
)

model = model.to(device)

# model = torch.compile(model)

In [10]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

# Add learning rate scheduler for better convergence
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=5,
    eta_min=1e-6
)

scaler = torch.cuda.amp.GradScaler()

In [19]:
# Clear GPU cache and check memory before training setup
torch.cuda.empty_cache()

print("=== GPU Memory Status ===")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print(f"Allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
print(f"Reserved: {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")
print(f"Free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0)) / 1024**3:.2f} GB")

=== GPU Memory Status ===
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Total VRAM: 8.00 GB
Allocated: 0.24 GB
Reserved: 0.59 GB
Free: 7.41 GB


In [11]:
def train_epoch(loader):

    model.train()

    running_loss = 0

    for images, labels in tqdm(loader):

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)  # More efficient than zero_grad()

        with torch.cuda.amp.autocast():

            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    return running_loss / len(loader)

In [12]:
def validate(loader):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in tqdm(loader):

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.cuda.amp.autocast():
                outputs = model(images)

            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [13]:
EPOCHS = 5

for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_loss = train_epoch(train_loader)

    val_acc = validate(val_loader)

    print("Train Loss:", train_loss)
    print("Validation Accuracy:", val_acc)
    print("Learning Rate:", optimizer.param_groups[0]['lr'])
    print(f"GPU Memory: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
    
    # Step the scheduler
    scheduler.step()
    
    # Clear cache after each epoch
    torch.cuda.empty_cache()


Epoch 1/5


  1%|█▋                                                                                                                  | 41/2813 [01:09<1:18:33,  1.70s/it]


KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), "efficientnet_b4_baseline.pth")

print("Model saved.")

In [ ]:
model = timm.create_model(
    "efficientnet_b4",
    pretrained=False,
    num_classes=2
)

model.load_state_dict(torch.load("efficientnet_b4_baseline.pth"))
model.to(device)

model.eval()